### Defining parameters

In [151]:
import pandas as pd

year = 2024
columns_to_keep = [
    "locatie_code",
    "grootheid_code",
    "parameter_code",
    "hoedanigheid_code",
    "eventdatum",
    "event_aquocode",
    "eenheid_code",
]
data_path = rf"C:\Ocean\Work\Projects\2026\ISC\Tools\Repositories\voorbeeld\isc_2023-2025\ISC_{year}.xlsx"
data_delivered_nl_path =  r"C:\Ocean\Work\Projects\2026\ISC\Tools\Repositories\voorbeeld\ISC-CIE WGM_Oct 2025_NL.xlsx"
data_delivered_fr_path = r"C:\Ocean\Work\Projects\2026\ISC\Tools\Repositories\voorbeeld\ISC-CIE WGM_Tranfert des données RHME 2024_Sept 2025.xlsx"

mapping_location_path = r"C:\Ocean\Work\Projects\2026\ISC\Tools\Repositories\mappings\locations-mapped.xlsx"
mapping_substances_path = r"C:\Ocean\Work\Projects\2026\ISC\Tools\Repositories\mappings\parameter_mapping_final.xlsx"
mapping_hoedanigheid_path = r"C:\Ocean\Work\Projects\2026\ISC\Tools\Repositories\mappings\hoedanigheid_mapped.xlsx"

df_data = pd.read_excel(data_path,sheet_name=str(year))
df_delivered_nl = pd.read_excel(data_delivered_nl_path, sheet_name="Paramètres - Parameters", header=1)
df_delivered_fr = pd.read_excel(data_delivered_fr_path, sheet_name="Paramètres - Parameters", header=1)

df_mapping_location = pd.read_excel(mapping_location_path)
df_mapping_substances = pd.read_excel(mapping_substances_path, sheet_name="mapping")
df_mapping_hoedanigheid = pd.read_excel(mapping_hoedanigheid_path)

In [152]:
# Keep record of non-measured parameters
df_mapping_nm = df_mapping_substances[df_mapping_substances["reported"].isin(["N"])]

# Remove from df_mapping_substances any rows where reported column is not nan
df_mapping_substances = df_mapping_substances[~df_mapping_substances["reported"].isin(["N", "S"])] 

### Defining functions

In [153]:
def mapping_location(df_data, df_mapping):
    # Mapping column names
    original_loc_col = "locatie_code"
    target_loc_col = "Identitication unique de la station"

    # Keep one mapping row per location code
    df_mapping = df_mapping[[original_loc_col, target_loc_col]]

    # Merge station unique ID into df_data on locatie_code
    df_data = df_data.merge(df_mapping, on=original_loc_col, how="left")

    missing_count = df_data[target_loc_col].isna().sum()
    print(f"Missing location data: {missing_count} rows")
    print(f"Shape of df_data after mapping location: {df_data.shape}")
    return df_data

def mapping_substances_by_combination(df_data, df_mapping):
    """
    Add 'Unieke identificatie gemeten parameter' to df_data by matching on:
    parameter_code, grootheid_code, hoedanigheid_code, eenheid_code
  Drop rows without a unique parameter id.
    """

    match_cols = [
        "parameter_code",
        "grootheid_code",
        "hoedanigheid_code",
        "eenheid_code",
    ]
    target_parameter_col = "Unieke identificatie gemeten parameter"

    df_mapping = df_mapping[match_cols + [target_parameter_col]].copy()

    for col in match_cols:
        df_mapping[col] = df_mapping[col].astype(str).str.strip()
        df_data[col] = df_data[col].astype(str).str.strip()

    duplicates = df_mapping.duplicated(subset=match_cols, keep=False)
    if duplicates.any():
        print(f"Warning: {duplicates.sum()} duplicate mapping rows on match columns")

    df_mapping = df_mapping.drop_duplicates(subset=match_cols, keep="first")
    print(f"Shape of df_mapping: {df_mapping.shape}")

    rows_before = len(df_data)

    # Left merge first, then drop rows without unique id
    df_data = df_data.merge(df_mapping, on=match_cols, how="left")
    df_data = df_data[df_data[target_parameter_col].notna()]

    rows_dropped = rows_before - len(df_data)
    print(f"Dropped rows without unique id: {rows_dropped}")
    print(f"Shape of df_data after mapping substances: {df_data.shape}")

    return df_data

def mapping_substances_by_combination(df_data, df_mapping):
    """
    Add 'Unieke identificatie gemeten parameter',
    'Unieke identificatie van de eenheid', and 'conversion'
    to df_data by matching on:
    parameter_code, grootheid_code, hoedanigheid_code, eenheid_code
    Drop rows without a unique parameter id.
    """

    match_cols = [
        "parameter_code",
        "grootheid_code",
        "hoedanigheid_code",
        "eenheid_code",
    ]
    target_cols = [
        "Unieke identificatie gemeten parameter",
        "Unieke identificatie van de eenheid",
        "conversion",
    ]

    df_mapping = df_mapping[match_cols + target_cols].copy()

    for col in match_cols:
        df_mapping[col] = df_mapping[col].astype(str).str.strip()
        df_data[col] = df_data[col].astype(str).str.strip()

    duplicates = df_mapping.duplicated(subset=match_cols, keep=False)
    if duplicates.any():
        print(f"Warning: {duplicates.sum()} duplicate mapping rows on match columns")

    df_mapping = df_mapping.drop_duplicates(subset=match_cols, keep="first")
    print(f"Shape of df_mapping: {df_mapping.shape}")

    rows_before = len(df_data)

    df_data = df_data.merge(df_mapping, on=match_cols, how="left")
    df_data = df_data[df_data["Unieke identificatie gemeten parameter"].notna()]

    rows_dropped = rows_before - len(df_data)
    print(f"Dropped rows without unique id: {rows_dropped}")
    print(f"Shape of df_data after mapping substances: {df_data.shape}")

    return df_data
    
def mapping_substances_by_aquocode(df_data, df_mapping):

    # ds column and equivalent mapping key
    original_parameter_col = "parameter_code"

    # Rename "AQUO_Codes" to "parameter_code"
    df_mapping = df_mapping.rename(columns={"AQUO_Codes": original_parameter_col})

    # Column you want to bring from mapping table
    target_parameter_col = "Unieke identificatie gemeten parameter"  

    df_mapping = df_mapping[[original_parameter_col, target_parameter_col]]
    print(f"Shape of df_mapping: {df_mapping.shape}")

    # # This keeps only rows where parameter exists in the mapping parameter table 
    # df_data = df_data.merge(df_mapping, on=original_parameter_col, how="inner")
    # This keeps all rows and adds the parameter column from the mapping table
    df_data = df_data.merge(df_mapping, on=original_parameter_col, how="left")
    print(f"Shape of df_data after mapping location: {df_data.shape}")
    return df_data

def mapping_substances_by_grootheid(df_data, df_mapping):
    """
    Fill missing 'Unieke identificatie gemeten parameter' values using grootheid_code
    for special grootheid codes only.
    """
    original_grootheid_col = "grootheid_code"
    target_parameter_col = "Unieke identificatie gemeten parameter"

    special_grootheid_codes = ["GELDHD", "HH", "T", "pH"]

    if original_grootheid_col not in df_data.columns:
        raise KeyError(f"Missing column in df_data: {original_grootheid_col}")
    if original_grootheid_col not in df_mapping.columns:
        raise KeyError(f"Missing column in df_mapping: {original_grootheid_col}")
    if target_parameter_col not in df_mapping.columns:
        raise KeyError(f"Missing column in df_mapping: {target_parameter_col}")

    # Keep only the special grootheid codes in the mapping table
    df_mapping = df_mapping[[original_grootheid_col, target_parameter_col]].copy()
    df_mapping = df_mapping[df_mapping[original_grootheid_col].isin(special_grootheid_codes)]

    # Avoid duplicate merge rows
    df_mapping = df_mapping.drop_duplicates(subset=[original_grootheid_col])

    print(f"Shape of grootheid mapping: {df_mapping.shape}")

    # Merge without overwriting existing parameter ids
    df_data = df_data.merge(
        df_mapping,
        on=original_grootheid_col,
        how="left",
        suffixes=("", "_grootheid"),
    )

    # Fill only rows that still do not have a unique parameter id
    df_data[target_parameter_col] = df_data[target_parameter_col].fillna(
        df_data[f"{target_parameter_col}_grootheid"]
    )

    # Drop helper column
    df_data = df_data.drop(columns=[f"{target_parameter_col}_grootheid"])

    print(f"Shape of df_data after grootheid mapping: {df_data.shape}")
    return df_data

def mapping_hoedanigheid(df_data, df_mapping):
    # Rename hoedanigheid_code_wadar to hoedanigheid_code in the mapping table
    df_mapping = df_mapping.rename(columns={"hoedanigheid_code_wadar": "hoedanigheid_code"})

    # Mapping column names
    original_hoedanigheid_col = "hoedanigheid_code"
    target_hoedanigheid_col = "ISC_fraction"

    # Keep one mapping row per hoedanigheid code
    df_mapping = df_mapping[[original_hoedanigheid_col, target_hoedanigheid_col]]

    # Merge hoedanigheid unique ID into df_data on hoedanigheid_code
    df_data = df_data.merge(df_mapping, on=original_hoedanigheid_col, how="left")
    print(f"Shape of df_data after mapping hoedanigheid: {df_data.shape}")
    return df_data

def drop_rows_without_parameter(df_data):
    target_parameter_col = "Unieke identificatie gemeten parameter"
    df_data = df_data[df_data[target_parameter_col].notna()]
    print(f"Shape of df_data after dropping rows without parameter: {df_data.shape}")
    return df_data

def filter_data(df_data, id_col="Unieke identificatie gemeten parameter"):
    ids_before = set(df_data[id_col].dropna().astype(str).unique())
    print(f"Parameter IDs at start: {len(ids_before)}")

    # Remove rows where waardebewerkings_methode_code is BER
    filtered_df = df_data[df_data["waardebewerkings_methode_code"] != "BER"]
    print("Shape of filtered DataFrame (excluding BER):", filtered_df.shape)

    # Keep only bemonsteringshoogte_code == -100
    filtered_df = filtered_df[filtered_df["bemonsteringshoogte_code"] == -100]
    print("Shape of filtered DataFrame (bemonsteringshoogte_code == -100):", filtered_df.shape)

    # Keep only valid event_aquocode values
    valid_aquocodes = [0, 3, 90, 99]
    filtered_df = filtered_df[filtered_df["event_aquocode"].isin(valid_aquocodes)]
    print("Shape of filtered DataFrame (valid event_aquocode):", filtered_df.shape)

    ids_after = set(filtered_df[id_col].dropna().astype(str).unique())
    ids_lost = sorted(ids_before - ids_after, key=lambda x: (not x.isdigit(), int(x) if x.isdigit() else x))

    print(f"Parameter IDs after filter: {len(ids_after)}")
    print(f"Parameter IDs lost by filter: {len(ids_lost)}")

    if ids_lost:
        print("Lost IDs:")
        print(ids_lost)
    else:
        print("No parameter IDs were lost during filtering.")

    return filtered_df

def select_lowest_aquocode_with_notes(df_data):
    """
    For rows with identical location, date, hoedanigheid, parameter, and eenheid,
    keep the one with the lowest valid aquocode and note alternative aquocodes.
    
    Parameters:
    -----------
    df_data : DataFrame
        Input dataframe (typically filtered_df)
    
    Returns:
    --------
    tuple : (kept_df, removed_df)
        kept_df: Dataframe with duplicate aquocodes resolved, plus new columns for alternatives
        removed_df: Dataframe with all rows that were removed/not selected
    """
    # Hardcoded column names
    target_loc_col = "Identitication unique de la station"
    target_hoedanigheid_col = "ISC_fraction"
    target_parameter_col = "Unieke identificatie gemeten parameter"
    
    valid_aquocodes = [0, 3, 90, 99]
    
    # Group by the common columns
    group_cols = [
        target_loc_col, 
        "eventdatum", 
        target_hoedanigheid_col, 
        target_parameter_col, 
        "eenheid_code"
    ]
    
    kept_rows = []
    removed_rows = []
    
    for group_key, group_df in df_data.groupby(group_cols, dropna=False):
        # Get all aquocodes in this group
        aquocodes_in_group = group_df["event_aquocode"].tolist()
        
        # Find the lowest valid aquocode
        valid_in_group = [ac for ac in aquocodes_in_group if ac in valid_aquocodes]
        lowest_aquocode = min(valid_in_group) if valid_in_group else None
        
        # Select row with lowest aquocode
        row_to_keep = group_df[group_df["event_aquocode"] == lowest_aquocode].iloc[0].copy()
        
        # Find higher aquocodes that were present
        higher_aquocodes = [ac for ac in valid_aquocodes if ac > lowest_aquocode]
        higher_present = {ac: aquocodes_in_group.count(ac) for ac in higher_aquocodes if ac in aquocodes_in_group}
        
        # Add columns for alternative aquocodes
        row_to_keep["has_higher_aquocodes"] = "yes" if higher_present else "no"
        row_to_keep["higher_aquocodes_info"] = str(higher_present) if higher_present else ""
        
        kept_rows.append(row_to_keep)
        
        # Add removed rows to removed_rows list
        rows_removed = group_df[group_df["event_aquocode"] != lowest_aquocode]
        if len(rows_removed) > 0:
            removed_rows.extend(rows_removed.to_dict('records'))
    
    kept_df = pd.DataFrame(kept_rows).reset_index(drop=True)
    removed_df = pd.DataFrame(removed_rows).reset_index(drop=True)
    
    print(f"Shape of result after selecting lowest aquocode: {kept_df.shape}")
    print(f"Shape of removed rows: {removed_df.shape}")
    
    return kept_df, removed_df

def get_count_unique_parameter_values(filtered_df):
    target_parameter_col = "Unieke identificatie gemeten parameter"
    # Get count of unique values in the target column after merge
    unique_values_after_merge = filtered_df[target_parameter_col].value_counts(dropna=False)
    print(unique_values_after_merge)
    print(f"Shape of unique_values_after_merge: {unique_values_after_merge.shape}")
    return unique_values_after_merge

def check_duplicates(filtered_df):
    target_loc_col = "Identitication unique de la station"
    target_parameter_col = "Unieke identificatie gemeten parameter"
    target_hoedanigheid_col = "ISC_fraction"
    # Keep only the columns of interest for duplicate checking
    columns_to_keep = [target_loc_col ,"grootheid_code",target_parameter_col , target_hoedanigheid_col, "eventdatum", "event_aquocode", "eenheid_code"]
    filtered_df_duplicates = filtered_df[columns_to_keep].copy()
    filtered_df_duplicates 

    # Check if there are duplicated rows in the filtered DataFrame
    has_identical_rows = filtered_df_duplicates.duplicated().any()
    print("Identical rows found:" if has_identical_rows else "No identical rows found.")

def split_by_station(df_data):
    target_loc_col = "Identitication unique de la station"
    dfs_by_station = {
        station_id: group_df.reset_index(drop=True)
        for station_id, group_df in df_data.groupby(target_loc_col, dropna=False)
    }
    print(f"Created {len(dfs_by_station)} station DataFrames")
    return dfs_by_station

def get_repeated_cases_by_station(dfs_by_station):
    target_loc_col = "Identitication unique de la station"
    target_parameter_col = "Unieke identificatie gemeten parameter"
    columns_to_keep = [
        target_loc_col,
        "grootheid_code",
        target_parameter_col,
        "ISC_fraction",
        "eventdatum",
        "event_aquocode",
        "eenheid_code",
    ]
    case_cols = [c for c in columns_to_keep if c != "eventdatum"]

    repeated_cases_by_station = {}

    for station_id, station_df in dfs_by_station.items():
        station_df_duplicates = station_df[columns_to_keep].copy()

        repeated_cases = (
            station_df_duplicates
            .groupby(case_cols, dropna=False)
            .agg(
                n_measurements=("eventdatum", "size"),
                n_unique_dates=("eventdatum", "nunique"),
                first_date=("eventdatum", "min"),
                last_date=("eventdatum", "max"),
            )
            .reset_index()
        )

        repeated_cases = repeated_cases[repeated_cases["n_measurements"] >= 1]
        repeated_cases = repeated_cases.sort_values("n_measurements", ascending=False)

        repeated_cases_by_station[station_id] = repeated_cases.reset_index(drop=True)

    print(f"Created {len(repeated_cases_by_station)} repeated-case tables")
    return repeated_cases_by_station

def harmonize_df(df_data):
    target_loc_col = "Identitication unique de la station"
    target_hoedanigheid_col = "ISC_fraction"
    target_parameter_col = "Unieke identificatie gemeten parameter"

    columns_to_keep = [
        target_loc_col,
        "eventdatum",
        target_hoedanigheid_col,
        target_parameter_col,
        "event_waarde_limietsymbool",
        "event_waarde",
        "eenheid_code",
        "Unieke identificatie van de eenheid",
        "conversion",
        "event_aquocode",
    ]

    missing_cols = [c for c in columns_to_keep if c not in df_data.columns]
    if missing_cols:
        raise KeyError(f"Missing required columns: {missing_cols}")

    result_df = df_data[columns_to_keep].copy()

    # Parse date once for reliable sorting and formatting
    result_df["eventdatum"] = pd.to_datetime(result_df["eventdatum"], errors="coerce")

    # Sort rows: location -> date -> substance(parameter)
    result_df = result_df.sort_values(
        by=[target_loc_col, "eventdatum", target_parameter_col],
        ascending=[True, True, True],
        na_position="last",
        kind="mergesort"
    ).reset_index(drop=True)

    # Format date as dd/mm/jjjj
    result_df["eventdatum"] = result_df["eventdatum"].dt.strftime("%d/%m/%Y")

    # Calculate result
    result_df["Résultat"] = result_df["event_waarde"]*result_df["conversion"]

    #if column event_waarde is numeric and "event_waarde_limietsymbool is NaN then make it = give me the line for result_df
    mask = pd.to_numeric(result_df["event_waarde"], errors="coerce").notna() & result_df["event_waarde_limietsymbool"].isna()
    result_df.loc[mask, "event_waarde_limietsymbool"] = "="


    # Harmonize output names
    rename_map = {
        target_loc_col: "Identitication unique de la station",
        "eventdatum": "Date de prélévement",
        target_hoedanigheid_col: "Fraction analysée",
        target_parameter_col: "Identification unique du paramètre mesuré",
        "event_waarde_limietsymbool": "Gestion de la Limite de Quantification",
        "Unieke identificatie van de eenheid": "Identification unique de l'unité",
    }
    result_df = result_df.rename(columns=rename_map)

    final_cols = [
        "Identitication unique de la station",
        "Date de prélévement",
        "Fraction analysée",
        "Identification unique du paramètre mesuré",
        "Gestion de la Limite de Quantification",
        "Résultat",
        "Identification unique de l'unité",
        "event_waarde",
        "eenheid_code",
        "conversion",
        "event_aquocode",
    ]
    result_df = result_df[final_cols]

    print(f"Shape of harmonized dataframe: {result_df.shape}")
    return result_df

def get_repeated_cases_by_station(dfs_by_station):
    target_loc_col = "Identitication unique de la station"
    target_parameter_col = "Unieke identificatie gemeten parameter"
    target_hoedanigheid_col = "ISC_fraction"
    columns_to_keep = [
        target_loc_col,
        target_parameter_col,
        target_hoedanigheid_col,
        "eventdatum",
        "event_aquocode",
        "eenheid_code",
    ]
    case_cols = [c for c in columns_to_keep if c != "eventdatum"]

    repeated_cases_by_station = {}

    for station_id, station_df in dfs_by_station.items():
        station_df_duplicates = station_df[columns_to_keep].copy()

        # normalize to calendar day (ignore time part)
        station_df_duplicates["event_day"] = pd.to_datetime(
            station_df_duplicates["eventdatum"], errors="coerce"
        ).dt.normalize()

        def same_day_duplicate_info(group):
            # count how many measurements per day within this case
            day_counts = group["event_day"].value_counts(dropna=True)

            # days with more than 1 measurement
            has_same_day = (day_counts > 1).any()

            # unique-parameter ids that appear on a duplicated day
            # (for this grouping, it will usually be the same id, but kept generic)
            if has_same_day:
                ids = sorted(group[target_parameter_col].dropna().astype(str).unique())
                ids_text = ", ".join(ids)
            else:
                ids_text = ""

            return pd.Series(
                {
                    "n_measurements": group["eventdatum"].size,
                    "n_unique_dates": group["eventdatum"].nunique(),
                    "first_date": group["eventdatum"].min(),
                    "last_date": group["eventdatum"].max(),
                    "same_day_measurement": "yes" if has_same_day else "no",
                    "same_day_parameter_ids": ids_text,
                }
            )
        
        value_cols = ["eventdatum", "event_day", target_parameter_col]

        repeated_cases = (
            station_df_duplicates
            .groupby(case_cols, dropna=False)[value_cols]
            .apply(same_day_duplicate_info)
            .reset_index()
        )

        repeated_cases = repeated_cases[repeated_cases["n_measurements"] >= 1]
        repeated_cases = repeated_cases.sort_values("n_measurements", ascending=False)

        repeated_cases_by_station[station_id] = repeated_cases.reset_index(drop=True)

    print(f"Created {len(repeated_cases_by_station)} repeated-case tables")
    return repeated_cases_by_station

def print_parameter_counts_by_event_aquocode(
    repeated_cases_by_station,
    valid_aquocodes=None,
    show_only_duplicates=True,
    top_n=None,
):
    if valid_aquocodes is None:
        valid_aquocodes = [0, 3, 90, 99]

    target_parameter_col = "Unieke identificatie gemeten parameter"
    target_hoedanigheid_col = "ISC_fraction"
    same_day_col = "same_day_measurement"  # expects values like "yes"/"no"

    grand_total_cases = 0
    grand_total_same_day_yes = 0

    def print_parameter_summary(df, indent="      "):
        counts = df[target_parameter_col].value_counts(ascending=False)

        if show_only_duplicates:
            counts = counts[counts > 1]

        if top_n is not None:
            counts = counts.head(top_n)

        if len(counts) == 0:
            print(f"{indent}No duplicate parameters")
            return

        for parameter_id, count in counts.items():
            print(f"{indent}parameter {parameter_id}: {count} cases")

    for station_id, station_df in repeated_cases_by_station.items():
        print("\n" + "=" * 70)
        print(f"STATION: {station_id}")
        print("=" * 70)

        station_total_cases = 0
        station_total_same_day_yes = 0

        for event_aquocode in valid_aquocodes:
            df_aquocode = station_df[station_df["event_aquocode"] == event_aquocode]
            n_cases = len(df_aquocode)
            n_unique_parameters = df_aquocode[target_parameter_col].nunique()
            station_total_cases += n_cases

            # Count rows/cases flagged as yes for same-day measurements
            n_same_day_yes = (
                df_aquocode[same_day_col]
                .astype(str)
                .str.lower()
                .eq("yes")
                .sum()
            )
            station_total_same_day_yes += n_same_day_yes

            print(f"\n  AQUOCODE {event_aquocode}")
            print(f"  Cases: {n_cases} | Unique parameters: {n_unique_parameters}")
            print(f"  Cases with same-day measurements (yes): {n_same_day_yes}")

            if n_cases == 0:
                print("  No rows")
                continue

            # Aquocode level
            print("  Summary (all hoedanigheid_code):")
            print_parameter_summary(df_aquocode, indent="    ")

            # Hoedanigheid level
            hoedanigheid_groups = (
                df_aquocode
                .groupby(target_hoedanigheid_col, dropna=False)
                .size()
                .sort_values(ascending=False)
            )

            print("  By hoedanigheid_code:")
            for hoedanigheid_code in hoedanigheid_groups.index:
                df_h = df_aquocode[df_aquocode[target_hoedanigheid_col] == hoedanigheid_code]
                n_cases_h = len(df_h)
                n_unique_h = df_h[target_parameter_col].nunique()
                n_same_day_yes_h = (
                    df_h[same_day_col]
                    .astype(str)
                    .str.lower()
                    .eq("yes")
                    .sum()
                )

                print(
                    f"    - {hoedanigheid_code}: "
                    f"{n_cases_h} cases, {n_unique_h} unique parameters, "
                    f"{n_same_day_yes_h} with same-day measurements (yes)"
                )
                print_parameter_summary(df_h, indent="      ")

        print(f"\n  STATION TOTAL: {station_total_cases} cases")
        print(f"  STATION TOTAL with same-day measurements (yes): {station_total_same_day_yes} cases")

        grand_total_cases += station_total_cases
        grand_total_same_day_yes += station_total_same_day_yes

    print("\n" + "=" * 70)
    print(f"GRAND TOTAL: {grand_total_cases} cases")
    print(f"GRAND TOTAL with same-day measurements (yes): {grand_total_same_day_yes} cases")
    print("=" * 70)

def aggregate_lq_symbol(series):
    """
    If all LQ symbols in a group are the same, return that value.
    If different, return a list and mark the group as inconsistent.
    """
    values = series.dropna().tolist()

    if not values:
        return None

    unique_values = list(dict.fromkeys(values))  # keep order, remove duplicates

    if len(unique_values) == 1:
        return unique_values[0]

    return unique_values

def compress_parameters(
    df,
    source_param_ids,
    target_param_id,
    group_cols=None,
    sum_cols=None,
    list_cols=None,
    remove_source_rows=True,
):
    source_param_ids = [str(x) for x in source_param_ids]
    target_param_id = str(target_param_id)
    expected_ids = set(source_param_ids)

    if group_cols is None:
        group_cols = [
            "Identitication unique de la station",
            "Date de prélévement",
            "Fraction analysée",
            "Identification unique de l'unité",
            "eenheid_code",
            "conversion",
        ]

    if sum_cols is None:
        sum_cols = ["Résultat", "event_waarde"]

    if list_cols is None:
        list_cols = ["event_aquocode"]

    lq_col = "Gestion de la Limite de Quantification"
    param_col = "Identification unique du paramètre mesuré"

    start_shape = df.shape

    print("=" * 70)
    print(f"Compressing {source_param_ids} -> {target_param_id}")
    print(f"remove_source_rows = {remove_source_rows}")
    print(f"Shape at start: {start_shape}")

    mask = df[param_col].astype(str).isin(source_param_ids)
    n_source_rows = int(mask.sum())
    df_sub = df[mask].copy()

    if df_sub.empty:
        print(f"No rows found for source_param_ids={source_param_ids}")
        print("=" * 70)
        return df.copy(), pd.DataFrame(), pd.DataFrame()

    print(f"Source rows found: {n_source_rows}")

  # --- incomplete source params ---
    incomplete_records = []

    for group_key, group_df in df_sub.groupby(group_cols, dropna=False):
        if not isinstance(group_key, tuple):
            group_key = (group_key,)

        available_ids = sorted(group_df[param_col].astype(str).unique().tolist())
        missing_ids = sorted(expected_ids - set(available_ids))

        if missing_ids:
            record = dict(zip(group_cols, group_key))
            record["target_param_id"] = target_param_id
            record["expected_source_params"] = source_param_ids
            record["available_source_params"] = available_ids
            record["missing_source_params"] = missing_ids
            record["n_available"] = len(available_ids)
            record["n_expected"] = len(source_param_ids)
            incomplete_records.append(record)

    df_incomplete_cases = pd.DataFrame(incomplete_records)

    if not df_incomplete_cases.empty:
        print(
            f"WARNING: {len(df_incomplete_cases)} incomplete case(s) found "
            f"for {source_param_ids} -> {target_param_id}"
        )
    else:
        print(
            f"All groups contain the full set of source params "
            f"({source_param_ids})"
        )

    # --- inconsistent LQ symbols ---
    lq_conflict_records = []

    for group_key, group_df in df_sub.groupby(group_cols, dropna=False):
        if not isinstance(group_key, tuple):
            group_key = (group_key,)

        lq_values = group_df[lq_col].dropna().tolist()
        unique_lq = list(dict.fromkeys(lq_values))

        if len(unique_lq) > 1:
            record = dict(zip(group_cols, group_key))
            record["target_param_id"] = target_param_id
            record["lq_values_found"] = unique_lq
            record["source_rows_in_group"] = len(group_df)
            lq_conflict_records.append(record)

    df_lq_conflicts = pd.DataFrame(lq_conflict_records)

    if not df_lq_conflicts.empty:
        print(
            f"WARNING: {len(df_lq_conflicts)} case(s) with different "
            f"'{lq_col}' symbols for {source_param_ids} -> {target_param_id}"
        )
    else:
        print(f"All '{lq_col}' symbols are consistent within groups")

    # --- aggregate ---
    agg_dict = {}

    for col in sum_cols:
        agg_dict[col] = (col, "sum")

    agg_dict[lq_col] = (lq_col, aggregate_lq_symbol)

    for col in list_cols:
        safe_name = col.replace(" ", "_")
        agg_dict[safe_name] = (col, lambda s: s.dropna().tolist())

    df_compressed = (
        df_sub.groupby(group_cols, dropna=False)
        .agg(**agg_dict)
        .reset_index()
    )

    rename_map = {col.replace(" ", "_"): col for col in list_cols}
    df_compressed = df_compressed.rename(columns=rename_map)

    df_compressed[param_col] = target_param_id

    col_order = [c for c in df.columns if c in df_compressed.columns]
    df_compressed = df_compressed[col_order]

    print(f"Compressed rows created: {len(df_compressed)}")
    print(f"Compression ratio: {n_source_rows} source rows -> {len(df_compressed)} compressed rows")

    if remove_source_rows:
        df_result = df[~mask].copy()
        print(f"Source rows removed: {n_source_rows}")
    else:
        df_result = df.copy()
        print("Source rows kept")

    print(f"Shape before appending compressed rows: {df_result.shape}")

    df_result = pd.concat([df_result, df_compressed], ignore_index=True)

    print(f"Compressed rows appended: {len(df_compressed)}")
    print(f"Shape after appending compressed rows: {df_result.shape}")
    print(f"Net change vs start: {df_result.shape[0] - start_shape[0]} rows")
    print("=" * 70)

    return df_result, df_incomplete_cases, df_lq_conflicts

def sort_harmonized_df(df):
    """
    Sort harmonized dataframe by:
    1. location
    2. substance (parameter id)
    3. date
    """
    sort_cols = [
        "Identitication unique de la station",
        "Identification unique du paramètre mesuré",
        "Date de prélévement",
    ]

    missing_cols = [c for c in sort_cols if c not in df.columns]
    if missing_cols:
        raise KeyError(f"Missing required columns: {missing_cols}")

    df_sorted = df.copy()

    # Sort safely even if date is text like '16/01/2024' or '16-1-2024'
    df_sorted["_sort_date"] = pd.to_datetime(
        df_sorted["Date de prélévement"],
        dayfirst=True,
        errors="coerce",
    )

    df_sorted = df_sorted.sort_values(
        by=[
            "Identitication unique de la station",
            "Identification unique du paramètre mesuré",
            "_sort_date",
        ],
        ascending=[True, True, True],
        na_position="last",
        kind="mergesort",
    ).drop(columns="_sort_date").reset_index(drop=True)

    return df_sorted

def format_result_4dec_and_set_nv_object(df, result_col="Résultat", aquo_col="event_aquocode"):
    out = df.copy()

    if aquo_col not in out.columns:
        raise KeyError(f"Missing required column: {aquo_col}")
    if result_col not in out.columns:
        raise KeyError(f"Missing required column: {result_col}")

    # Convert to numeric where possible
    numeric_vals = pd.to_numeric(out[result_col], errors="coerce")

    # Keep original values as object (for existing text like NV, etc.)
    out[result_col] = out[result_col].astype(object)

    # Write formatted numbers (4 decimals) as text
    num_mask = numeric_vals.notna()
    out.loc[num_mask, result_col] = numeric_vals.loc[num_mask].map(lambda x: f"{x:.4f}")

    # Force NV when aquocode is 99
    mask_99 = pd.to_numeric(out[aquo_col], errors="coerce").eq(99)
    out.loc[mask_99, result_col] = "NV"

    # Ensure dtype is object
    out[result_col] = out[result_col].astype(object)

    return out


def keep_only_final_columns(harmonized_df_final):
    cols_to_keep = [
        "Identitication unique de la station",
        "Date de prélévement",
        "Fraction analysée",
        "Identification unique du paramètre mesuré",
        "Gestion de la Limite de Quantification",
        "Résultat",
        "Identification unique de l'unité",
    ]

    missing = [c for c in cols_to_keep if c not in harmonized_df_final.columns]
    if missing:
        raise KeyError(f"Missing required columns: {missing}")

    return harmonized_df_final.loc[:, cols_to_keep].copy()

def rename_to_dutch_output(df):
    rename_map = {
        "Identitication unique de la station": "Unieke identiticatie meetpunt",
        "Fraction analysée": "Geanalyseerde fractie",
        "Date de prélévement": "Datum staalname",
        "Gestion de la Limite de Quantification": "Aanpak kwantificeringsgrens",
        "Résultat": "Resultaat",
        "Identification unique du paramètre mesuré": "Unieke identificatie gemeten parameter",
        "Identification unique de l'unité": "Unieke identificatie van de eenheid"
    }

    missing = [c for c in rename_map if c not in df.columns]
    if missing:
        raise KeyError(f"Missing required columns for rename: {missing}")

    out = df.rename(columns=rename_map).copy()

    # Keep only the Dutch output columns in the requested order
    dutch_cols = [
        "Unieke identiticatie meetpunt",
        "Geanalyseerde fractie",
        "Datum staalname",
        "Aanpak kwantificeringsgrens",
        "Resultaat",
        "Unieke identificatie gemeten parameter",
        "Unieke identificatie van de eenheid"
    ]
    return out.loc[:, dutch_cols]

def build_nm_table_from_mapping(
    df_mapping_nm,
    harmonized_df_final,
    default_meetpunt="",
    default_fractie="",
    default_datum="",
    default_lq="="
):
    param_col = "Unieke identificatie gemeten parameter"
    unit_col = "Unieke identificatie van de eenheid"

    required_map_cols = [param_col, unit_col]
    missing_map = [c for c in required_map_cols if c not in df_mapping_nm.columns]
    if missing_map:
        raise KeyError(f"Missing columns in df_mapping_nm: {missing_map}")

    target_cols = [
        "Unieke identiticatie meetpunt",
        "Geanalyseerde fractie",
        "Datum staalname",
        "Resultaat",
        "Unieke identificatie gemeten parameter",
        "Unieke identificatie van de eenheid",
    ]
    missing_target = [c for c in target_cols if c not in harmonized_df_final.columns]
    if missing_target:
        raise KeyError(f"Missing columns in harmonized_df_final: {missing_target}")

    nm_table = (
        df_mapping_nm[[param_col, unit_col]]
        .dropna(subset=[param_col])
        .drop_duplicates()
        .copy()
    )

    nm_table["Unieke identiticatie meetpunt"] = default_meetpunt
    nm_table["Geanalyseerde fractie"] = default_fractie
    nm_table["Datum staalname"] = default_datum
    nm_table["Resultaat"] = "NM"

    nm_table = nm_table[target_cols]
    return nm_table

### Running tasks

In [154]:
# df_data = mapping_location(df_data, df_mapping_location)
# df_data = mapping_substances_by_aquocode(df_data, df_mapping_substances)
# df_data = mapping_substances_by_grootheid(df_data, df_mapping_substances)
# df_data = mapping_hoedanigheid(df_data, df_mapping_hoedanigheid)
# df_data = drop_rows_without_parameter(df_data)
# filtered_df = filter_data(df_data)
# filtered_df, removed_df = select_lowest_aquocode_with_notes(filtered_df)

# # Select filtered_df where higher_aquocodes_info is not empty (i.e. there were higher aquocodes present for the same case)  
# filtered_df_with_higher_aquocodes = filtered_df[filtered_df["higher_aquocodes_info"] != ""]

# unique_values_after_merge = get_count_unique_parameter_values(filtered_df)
# check_duplicates(filtered_df)
# harmonized_df = harmonized_df(filtered_df)

# # Explore results 
# dfs_by_station = split_by_station(filtered_df)
# repeated_cases_by_station = get_repeated_cases_by_station(dfs_by_station)
# print_parameter_counts_by_event_aquocode(repeated_cases_by_station)

In [155]:
df_data = mapping_substances_by_combination(df_data, df_mapping_substances)
df_data = mapping_location(df_data, df_mapping_location)
df_data = mapping_hoedanigheid(df_data, df_mapping_hoedanigheid)
filtered_df = filter_data(df_data)
filtered_df, removed_df = select_lowest_aquocode_with_notes(filtered_df)
# filtered_df[filtered_df["has_higher_aquocodes"]!="no"]
harmonized_df = harmonize_df(filtered_df)
# filtered_df[filtered_df["Unieke identificatie gemeten parameter"]==1551]

df_compressed, incomplete_1774, lq_conflicts_1774 = compress_parameters(
    harmonized_df,
    source_param_ids=["1283", "1629", "1630"],
    target_param_id="1774",
    remove_source_rows=False,
)

df_compressed, incomplete_5534, lq_conflicts_5534 = compress_parameters(
    df_compressed,
    source_param_ids=["1103", "1181", "1173", "1207"],
    target_param_id="5534",
    remove_source_rows=False,
)

df_compressed, incomplete_6561, lq_conflicts_6561 = compress_parameters(
    df_compressed,
    source_param_ids=["6561a", "6561b"],
    target_param_id="6561",
    remove_source_rows=True,
)

harmonized_df_final = sort_harmonized_df(df_compressed)

harmonized_df_final = format_result_4dec_and_set_nv_object(harmonized_df_final) 
harmonized_df_final = keep_only_final_columns(harmonized_df_final)
harmonized_df_final = rename_to_dutch_output(harmonized_df_final)

Shape of df_mapping: (101, 7)
Dropped rows without unique id: 24119
Shape of df_data after mapping substances: (9284, 47)
Missing location data: 0 rows
Shape of df_data after mapping location: (9284, 48)
Shape of df_data after mapping hoedanigheid: (9284, 49)
Parameter IDs at start: 99
Shape of filtered DataFrame (excluding BER): (9284, 49)
Shape of filtered DataFrame (bemonsteringshoogte_code == -100): (6863, 49)
Shape of filtered DataFrame (valid event_aquocode): (6863, 49)
Parameter IDs after filter: 99
Parameter IDs lost by filter: 0
No parameter IDs were lost during filtering.
Shape of result after selecting lowest aquocode: (6861, 51)
Shape of removed rows: (2, 49)
Shape of harmonized dataframe: (6861, 11)
Compressing ['1283', '1629', '1630'] -> 1774
remove_source_rows = False
Shape at start: (6861, 11)
Source rows found: 189
All groups contain the full set of source params (['1283', '1629', '1630'])
All 'Gestion de la Limite de Quantification' symbols are consistent within group

In [156]:
#%% work with data in different format
# script to prep the chlorophyl-a data. Is prepped and works. 
import pandas as pd
import os
from pathlib import Path
p=Path(os.getcwd())

schaarvd = pd.read_excel(Path.joinpath(p.parent, 
                                       'voorbeeld/isc_2023-2025/SCHAARVODDL + SASVGT_CHLfa_2023-2025.xlsx'), 
                                       sheet_name='SCHAARVODDL CHLfa')
sasvgt = pd.read_excel(Path.joinpath(p.parent, 
                                       'voorbeeld/isc_2023-2025/SCHAARVODDL + SASVGT_CHLfa_2023-2025.xlsx'), 
                                       sheet_name='SASVGT CHLfa')


locatie = pd.DataFrame({
    'iscformat': ['NL89_SASVGT','NL89_SCHAARVODDL'], 
    'rwsformat': ['SASVGT','SCHAARVODDL']
})

columns = ["uid", "PARAMETRE", "n° CAS nr", 
        "Unieke identificatie gemeten parameter", "Unieke eenheidsidentificatie", 'ComponentName']
values = [1439, 'Chlorophylle a', '479-61-8', 'Chlorofyl a', 'µg/L','CHLFa'] ## quick fix to chlorofyl-a. if other parameters are used, this needs to be adjusted in the function

parameter_df = pd.DataFrame([values], columns=columns)
#hoedanigheid is only NVT -> EB in this dataset
# make a function to apply to both datasets, first fix the data before applying the function
def run_function_on_data(df, locatie, parameter):
    cols = [ 'ComponentName', 'UHoedanigheid', 'ResultType', 'UMeetpunt',
        'ResultText', 'ResultUMeetonzekerheid', 'ResultValue','UGeplandeDatum'] 
    df = df[cols]
    meetpunt = df['UMeetpunt'].unique()
    locatie_code = locatie[locatie['rwsformat']==meetpunt[0]]
    df['Unieke identiticatie meetpunt'] = df['UMeetpunt'].replace(meetpunt, locatie_code['iscformat'].values[0])
    df['Geanalyseerde fractie'] = df['UHoedanigheid'].replace('NVT', 'EB')
    df['Datum staalname'] =  pd.to_datetime(df['UGeplandeDatum'], format="mixed").dt.strftime('%d/%m/%Y') 
    df['kwantificeringsgrens_bool'] = df['ResultText'].str.contains('<', regex=True)
    df['Aanpak kwantificeringsgrens'] = df['kwantificeringsgrens_bool'].apply(lambda x: '<' if x else '=')
    df['Resultaat'] = df['ResultValue'].replace(-999.0, 'NV')
    df['Unieke identificatie gemeten parameter'] = parameter["uid"].values[0]
    df['Unieke identificatie van de eenheid'] = "µg/L"
    final_cols = ['Unieke identiticatie meetpunt', 'Geanalyseerde fractie',
       'Datum staalname', 
       'Aanpak kwantificeringsgrens', 'Resultaat',
       'Unieke identificatie gemeten parameter', 'Unieke identificatie van de eenheid']
    
    df = df[final_cols]
    return df
# in resulttype zit de kwantificeringsgrens + aquo_code, onderverdelen in kwantificeringsgrens + wat de waare is
# %%r
schaar_output= run_function_on_data(schaarvd, locatie, parameter_df)
# %%
sasvgt_output = run_function_on_data(sasvgt, locatie, parameter_df)
# %%
df_chl = pd.concat([schaar_output, sasvgt_output], ignore_index=True)

df_chl_output_year = (
    df_chl[
        pd.to_datetime(df_chl["Datum staalname"], dayfirst=True).dt.year == year
    ]
    .reindex(columns=harmonized_df_final.columns)
)

In [157]:
harmonized_df_with_chl = pd.concat(
    [harmonized_df_final, df_chl_output_year],
    ignore_index=True
)

nm_table = build_nm_table_from_mapping(df_mapping_nm, harmonized_df_with_chl )

harmonized_df_with_chl_and_nm  = pd.concat(
    [harmonized_df_with_chl, nm_table],
    ignore_index=True
)

# Save the harmonized dataframe to an Excel file
output_path = rf"C:\Ocean\Work\Projects\2026\ISC\Tools\Repositories\voorbeeld\isc_2023-2025\ISC_{year}_harmonized.csv"
harmonized_df_with_chl_and_nm .to_csv(output_path, index=False, sep=";", encoding="utf-8-sig")